# Lesson03. 机器狗出发！让它指哪走哪

**教学主题：** 学习控制机器人的精准移动。

**核心目标：** 理解简单的二维坐标概念。用代码控制机器人前进、后退与转向。

**课程安排：**

- **前20分钟（概念课）：** 介绍高级运动指令 `move(x, y, yaw)`。用地面上的十字胶带做比喻，理解x（前后）、y（左右）、yaw（旋转）三个方向。

- **后100分钟（路径挑战）：**
  - **直线冲刺：** 编写脚本，让Go2向前走2米后停下。
  - **完美转身：** 让Go2原地旋转90度、180度。
  - **综合任务【挑战】：** 编写程序，让Go2在地板上走一个正方形路径。

## 3.1 导入依赖并初始化客户端

和Lesson02一样，先导入SDK并建立与机器狗的连接。

In [ ]:
import time  # 时间模块，用于控制延时
import sys   # 系统模块

# 导入宇树SDK通信和运动控制模块
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化通信通道
ChannelFactoryInitialize(0, "ens37")

# 创建并初始化运动控制客户端
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()

### 3.1.1 向前移动

调用 `Move(vx, vy, vyaw)` 方法控制机器狗运动：
- `vx`：前后方向速度（正值前进，负值后退），单位 m/s
- `vy`：左右方向速度（正值向左，负值向右），单位 m/s
- `vyaw`：旋转角速度（正值逆时针，负值顺时针），单位 rad/s

In [ ]:
# 向前移动：vx=0.3 m/s，vy=0，vyaw=0（只向前，不侧移，不转弯）
ret = sport_client.Move(0.3, 0, 0)
print("返回值: ", ret)  # 返回0表示成功

### 3.1.2 向左移动

设置 `vy` 为正值，机器狗将向左侧移。

In [ ]:
# 向左侧移：vx=0，vy=0.3 m/s，vyaw=0
sport_client.Move(0, 0.3, 0)

### 3.1.3 原地转弯

设置 `vyaw` 为正值，机器狗将逆时针旋转。

In [ ]:
# 原地转弯：vx=0，vy=0，vyaw=0.5 rad/s（逆时针旋转）
sport_client.Move(0, 0, 0.5)

## 3.2 挑战：走正方形路径

综合运用前进和侧移指令，让机器狗在地面上走出一个正方形。

**思路：** 依次沿四个方向移动相同的距离——先向前，再向左，再向后，最后向右，即可形成一个正方形路径。

In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化通信通道和运动控制客户端
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()


def move_continuous(v_x, v_y, duration, interval=0.4):
    """
    持续发送运动指令，使机器人按指定速度运动指定时长。
    
    由于单次Move指令的有效时间约为0.5秒，需要循环发送指令来实现持续运动。
    
    Args:
        v_x (float): x方向速度（前后），单位 m/s
        v_y (float): y方向速度（左右），单位 m/s
        duration (float): 总运动时长（秒）
        interval (float): 发送指令的时间间隔（秒），需小于单次指令有效时长0.5秒
    """
    start_time = time.time()
    while time.time() - start_time < duration:
        sport_client.Move(v_x, v_y, 0)
        time.sleep(interval)
    # 发送零速度指令，停止运动
    sport_client.Move(0, 0, 0)
    time.sleep(0.5)  # 等待停止指令生效


def move_square(base_vx=0.3, base_vy=0.3, scale=1.0, base_time=2.0):
    """
    控制机器人走正方形路径。
    
    通过依次沿x正方向、y正方向、x负方向、y负方向移动，
    形成一个闭合的正方形轨迹。
    
    Args:
        base_vx (float): x方向基础速度（m/s）
        base_vy (float): y方向基础速度（m/s）
        scale (float): 正方形边长比例因子，值越大边长越长
        base_time (float): 基础运动时间（秒），与scale相乘得到每条边的实际运动时间
    """
    # 计算每条边的运动时长
    edge_duration = base_time * scale
    
    # 第一条边：向前移动（x正方向）
    move_continuous(base_vx, 0, edge_duration)
    
    # 第二条边：向左移动（y正方向）
    move_continuous(0, base_vy, edge_duration)
    
    # 第三条边：向后移动（x负方向）
    move_continuous(-base_vx, 0, edge_duration)
    
    # 第四条边：向右移动（y负方向）
    move_continuous(0, -base_vy, edge_duration)


if __name__ == "__main__":
    # 设置正方形大小的比例因子，scale越大，正方形边长越长
    square_scale = 1.0
    move_square(scale=square_scale)

## 3.3 运动坐标系统详解


### 3.3.1 三维运动坐标系

Go2是一个在二维平面上运动的四足机器人（不涉及上下运动）。我们用三个参数控制它的运动：

```
      +Y (左)
       ↑
       |
  ←----+----→ +X (前)
       |
       ↓
      -Y (右)

顶视图：Go2从上往下看的样子

vyaw (旋转)：
  ↻ 正值（逆时针）
  ↺ 负值（顺时针）
```

### 3.3.2 速度参数详解

| 参数 | 范围 | 说明 | 例子 |
|------|------|------|------|
| `vx` | -3.5 ~ 3.5 | 前后速度（m/s） | 0.5=半速前进，-0.5=后退 |
| `vy` | -2.0 ~ 2.0 | 左右速度（m/s） | 0.3=向左移动，-0.3=向右移动 |
| `vyaw` | -2.0 ~ 2.0 | 旋转速度（rad/s） | 0.5=逆时针，-0.5=顺时针 |

**注意：** 这些都是最大值，实际使用时可以设置更小的值以获得更精确的控制。



### 3.3.3 距离与时间的关系

```
距离 = 速度 × 时间

如果要让Go2走1米：
  距离 = 1 m
  速度 = 0.5 m/s
  所需时间 = 1 / 0.5 = 2 秒

如果要让Go2转过90度（π/2 弧度）：
  角度 = π/2 rad
  角速度 = 0.5 rad/s
  所需时间 = (π/2) / 0.5 ≈ 3.14 秒
```

### 3.3.4 常用速度和时间组合


In [ ]:
# 走指定距离
def walk_distance(sport_client, distance, speed=0.3):
    """让Go2走指定距离"""
    duration = distance / speed
    move_continuous(sport_client, speed, 0, duration)

# 转过指定角度
def turn_angle(sport_client, angle_rad, angular_speed=0.5):
    """让Go2转过指定角度（弧度制）"""
    duration = angle_rad / angular_speed
    move_continuous(sport_client, 0, 0, duration, v_yaw=angular_speed)

# 常见的转角
import math
TURN_90 = math.pi / 2      # 90度转向
TURN_180 = math.pi          # 180度转向
TURN_360 = 2 * math.pi      # 360度转向（转圈）

## 3.4 常用运动模式

### 3.4.1 直线运动







In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()

# 等待机器人站立
sport_client.StandUp()
time.sleep(2)

# 向前走1米（速度0.5 m/s，所需时间2秒）
print("向前走1米...")
sport_client.Move(0.5, 0, 0)
time.sleep(2)
sport_client.Move(0, 0, 0)  # 停止
time.sleep(0.5)

# 向后走0.5米（负速度）
print("向后走0.5米...")
sport_client.Move(-0.3, 0, 0)
time.sleep(1.67)  # 0.5 / 0.3 ≈ 1.67秒
sport_client.Move(0, 0, 0)  # 停止
time.sleep(0.5)

print("直线运动完成！")

### 3.4.2 转向运动


In [ ]:
import math

# 原地转90度（逆时针）
print("原地转90度...")
angle = math.pi / 2  # 90度用弧度表示
angular_speed = 0.5  # rad/s
duration = angle / angular_speed

sport_client.Move(0, 0, angular_speed)
time.sleep(duration)
sport_client.Move(0, 0, 0)  # 停止
time.sleep(0.5)

# 原地转180度
print("原地转180度...")
angle = math.pi  # 180度
duration = angle / angular_speed
sport_client.Move(0, 0, angular_speed)
time.sleep(duration)
sport_client.Move(0, 0, 0)
time.sleep(0.5)

# 原地转圈（360度）
print("原地转圈...")
angle = 2 * math.pi  # 360度
duration = angle / angular_speed
sport_client.Move(0, 0, angular_speed)
time.sleep(duration)
sport_client.Move(0, 0, 0)
time.sleep(0.5)

print("转向运动完成！")

### 3.4.3 侧向移动


In [ ]:
# 向左走1米
print("向左走1米...")
speed_y = 0.3  # m/s
duration = 1.0 / speed_y  # 需要3.33秒

sport_client.Move(0, speed_y, 0)
time.sleep(duration)
sport_client.Move(0, 0, 0)
time.sleep(0.5)

# 向右走1米（负的vy）
print("向右走1米...")
sport_client.Move(0, -speed_y, 0)
time.sleep(duration)
sport_client.Move(0, 0, 0)
time.sleep(0.5)

print("侧向运动完成！")

### 3.4.4 复合运动（斜向移动）


In [ ]:
# 斜向前进（同时向前和向左）
print("斜向前进（向左前方）...")
sport_client.Move(0.3, 0.2, 0)  # 向前0.3 m/s，向左0.2 m/s
time.sleep(3)
sport_client.Move(0, 0, 0)
time.sleep(0.5)

# 斜向移动同时旋转
print("边走边转...")
sport_client.Move(0.2, 0, 0.3)  # 向前走，同时逆时针旋转
time.sleep(4)
sport_client.Move(0, 0, 0)
time.sleep(0.5)

print("复合运动完成！")

## 3.5 高级路径规划

首先导入必要的库并完成初始化和基本的函数定义：

In [ ]:
import time
import math
from unitree_sdk2py.core.channel import ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()

def move_distance(vx, vy, distance, speed=None):
    """移动指定距离"""
    if speed is None:
        speed = abs(vx) if vx != 0 else abs(vy)
    duration = distance / speed
    sport_client.Move(vx, vy, 0)
    time.sleep(duration)
    sport_client.Move(0, 0, 0)
    time.sleep(0.3)

def turn_angle(angle_rad, angular_speed=0.5):
    """转过指定角度"""
    duration = abs(angle_rad) / angular_speed
    direction = 1 if angle_rad > 0 else -1
    sport_client.Move(0, 0, direction * angular_speed)
    time.sleep(duration)
    sport_client.Move(0, 0, 0)
    time.sleep(0.3)


### 3.5.1 三角形路径

In [ ]:
# 走等边三角形（每边1米，每个角60度）
print("走等边三角形...")
sport_client.StandUp()
time.sleep(2)

edge_length = 1.0  # 1米
interior_angle = math.pi / 3  # 60度
exterior_angle = math.pi - interior_angle  # 120度

for i in range(3):
    print(f"第 {i+1} 条边...")
    move_distance(0.3, 0, edge_length)
    print(f"转过 {math.degrees(exterior_angle):.0f} 度...")
    turn_angle(exterior_angle)

print("✓ 等边三角形走完！")

### 3.5.2 五边形路径

In [ ]:
# 走正五边形（每边1米，每个角108度）
print("走正五边形...")
sport_client.StandUp()
time.sleep(2)

edge_length = 1.0
interior_angle = (5 - 2) * math.pi / 5  # 108度
exterior_angle = math.pi - interior_angle  # 72度

for i in range(5):
    print(f"第 {i+1} 条边...")
    move_distance(0.3, 0, edge_length)
    print(f"转过 {math.degrees(exterior_angle):.0f} 度...")
    turn_angle(exterior_angle)

print("✓ 正五边形走完！")

### 3.5.3 圆形路径（逼近）


In [ ]:
# 通过多个小直线段逼近圆形
print("走圆形...")
sport_client.StandUp()
time.sleep(2)

radius = 1.0  # 1米半径
num_segments = 20  # 用20条直线段逼近圆形

for i in range(num_segments):
    # 计算每条边的长度
    angle_segment = 2 * math.pi / num_segments
    segment_length = 2 * radius * math.sin(angle_segment / 2)
    
    print(f"第 {i+1}/{num_segments} 段...")
    move_distance(0.2, 0, segment_length)
    turn_angle(angle_segment)

print("✓ 圆形走完！")

### 3.5.4 8字形路径

In [ ]:
# 走8字形（两个相邻的圆形）
print("走8字形...")
sport_client.StandUp()
time.sleep(2)

radius = 0.8
num_segments = 16

# 第一个圆形
print("第一个圆形...")
for i in range(num_segments):
    angle_segment = 2 * math.pi / num_segments
    segment_length = 2 * radius * math.sin(angle_segment / 2)
    move_distance(0.2, 0, segment_length)
    turn_angle(angle_segment)

# 转向进入第二个圆形
turn_angle(math.pi)

# 第二个圆形（反向）
print("第二个圆形...")
for i in range(num_segments):
    angle_segment = 2 * math.pi / num_segments
    segment_length = 2 * radius * math.sin(angle_segment / 2)
    move_distance(0.2, 0, segment_length)
    turn_angle(angle_segment)

print("✓ 8字形走完！")

## 3.6 完整的工程实现

创建文件 `smart_navigator.py`：

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
智能导航系统 - Go2 路径规划与执行

这是一个完整的工程实现，包含：
1. 高级路径规划功能（正方形、三角形、圆形等）
2. 精确的距离和角度控制
3. 详细的日志和验证
"""

import time
import math
import sys
from unitree_sdk2py.core.channel import ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient


class SmartNavigator:
    """Go2智能导航系统"""
    
    def __init__(self, nic_name="ens37", timeout=10.0):
        """初始化导航系统"""
        self.nic_name = nic_name
        self.timeout = timeout
        self.sport_client = None
        self.total_distance = 0  # 总里程
        self.total_angle = 0  # 总转向角度
        
    def connect(self):
        """连接到机器人"""
        print(f"[连接] 使用网卡 {self.nic_name} 连接到 Go2...")
        try:
            ChannelFactoryInitialize(0, self.nic_name)
            self.sport_client = SportClient()
            self.sport_client.SetTimeout(self.timeout)
            self.sport_client.Init()
            print("✓ 连接成功！")
            return True
        except Exception as e:
            print(f"✗ 连接失败: {e}")
            return False
    
    def move_distance(self, distance, speed=0.3, direction="forward"):
        """
        移动指定距离
        
        参数：
            distance: 距离（米）
            speed: 速度（m/s）
            direction: 方向（forward/backward/left/right）
        """
        duration = distance / speed
        
        direction_map = {
            "forward": (speed, 0),
            "backward": (-speed, 0),
            "left": (0, speed),
            "right": (0, -speed)
        }
        
        if direction not in direction_map:
            print(f"✗ 未知方向: {direction}")
            return False
        
        vx, vy = direction_map[direction]
        
        print(f"[运动] 向{direction}走 {distance:.2f}m，速度 {speed} m/s，耗时 {duration:.2f}s")
        
        try:
            self.sport_client.Move(vx, vy, 0)
            time.sleep(duration)
            self.sport_client.Move(0, 0, 0)  # 停止
            time.sleep(0.3)
            
            self.total_distance += distance
            print(f"✓ 运动完成，累计里程: {self.total_distance:.2f}m")
            return True
        except Exception as e:
            print(f"✗ 运动失败: {e}")
            return False
    
    def turn_angle(self, angle_rad, angular_speed=0.5):
        """
        转过指定角度
        
        参数：
            angle_rad: 角度（弧度）
            angular_speed: 角速度（rad/s）
        """
        duration = abs(angle_rad) / angular_speed
        direction = 1 if angle_rad > 0 else -1
        angle_deg = abs(angle_rad) * 180 / math.pi
        
        print(f"[转向] 转过 {angle_deg:.1f}°，耗时 {duration:.2f}s")
        
        try:
            self.sport_client.Move(0, 0, direction * angular_speed)
            time.sleep(duration)
            self.sport_client.Move(0, 0, 0)  # 停止
            time.sleep(0.3)
            
            self.total_angle += abs(angle_rad)
            print(f"✓ 转向完成，累计转向: {self.total_angle * 180 / math.pi:.1f}°")
            return True
        except Exception as e:
            print(f"✗ 转向失败: {e}")
            return False
    
    def trace_polygon(self, sides, edge_length, speed=0.3):
        """
        走正多边形
        
        参数：
            sides: 边数（3=三角形，4=正方形，5=五边形...）
            edge_length: 每条边的长度（米）
            speed: 运动速度（m/s）
        """
        print(f"\n{'='*60}")
        print(f"【走正{sides}边形】")
        print(f"{'='*60}")
        
        interior_angle = (sides - 2) * math.pi / sides
        exterior_angle = math.pi - interior_angle
        
        try:
            for i in range(sides):
                print(f"\n第 {i+1}/{sides} 条边...")
                self.move_distance(edge_length, speed)
                
                if i < sides - 1:  # 最后一条边后不转向
                    self.turn_angle(exterior_angle)
            
            print(f"\n✓ 正{sides}边形走完！")
            return True
        except Exception as e:
            print(f"\n✗ 走多边形失败: {e}")
            return False
    
    def trace_rectangle(self, length, width, speed=0.3):
        """
        走矩形（通用的长方形路径）
        
        参数：
            length: 长（米）
            width: 宽（米）
            speed: 速度（m/s）
        """
        print(f"\n{'='*60}")
        print(f"【走矩形】({length}m × {width}m)")
        print(f"{'='*60}")
        
        try:
            # 第一条长边
            self.move_distance(length, speed)
            self.turn_angle(math.pi / 2)
            
            # 第一条宽边
            self.move_distance(width, speed)
            self.turn_angle(math.pi / 2)
            
            # 第二条长边
            self.move_distance(length, speed)
            self.turn_angle(math.pi / 2)
            
            # 第二条宽边
            self.move_distance(width, speed)
            
            print(f"\n✓ 矩形走完！")
            return True
        except Exception as e:
            print(f"\n✗ 走矩形失败: {e}")
            return False
    
    def trace_circle(self, radius, num_segments=20, speed=0.2):
        """
        走圆形（通过多个直线段逼近）
        
        参数：
            radius: 半径（米）
            num_segments: 分段数
            speed: 速度（m/s）
        """
        print(f"\n{'='*60}")
        print(f"【走圆形】(半径={radius}m)")
        print(f"{'='*60}")
        
        try:
            for i in range(num_segments):
                angle_segment = 2 * math.pi / num_segments
                segment_length = 2 * radius * math.sin(angle_segment / 2)
                
                print(f"第 {i+1}/{num_segments} 段...")
                self.move_distance(segment_length, speed)
                self.turn_angle(angle_segment)
            
            print(f"\n✓ 圆形走完！")
            return True
        except Exception as e:
            print(f"\n✗ 走圆形失败: {e}")
            return False
    
    def print_summary(self):
        """打印统计信息"""
        total_turns = self.total_angle * 180 / math.pi
        print(f"\n{'='*60}")
        print(f"【导航统计信息】")
        print(f"{'='*60}")
        print(f"总里程: {self.total_distance:.2f} 米")
        print(f"总转向: {total_turns:.1f}°")
        print(f"{'='*60}\n")


def main():
    """主程序"""
    
    # 创建导航系统
    navigator = SmartNavigator(nic_name="ens37")
    
    # 连接到机器人
    if not navigator.connect():
        return False
    
    # 启动机器人
    print("\n[启动] 让Go2站立...")
    navigator.sport_client.StandUp()
    time.sleep(2)
    
    print("\n" + "="*60)
    print("准备执行多条路径")
    print("="*60)
    
    try:
        # 路径1：正方形
        navigator.trace_polygon(sides=4, edge_length=1.0, speed=0.3)
        time.sleep(2)
        
        # 路径2：矩形
        navigator.trace_rectangle(length=1.5, width=0.8, speed=0.3)
        time.sleep(2)
        
        # 路径3：等边三角形
        navigator.trace_polygon(sides=3, edge_length=1.2, speed=0.3)
        time.sleep(2)
        
        # 路径4：圆形
        navigator.trace_circle(radius=0.8, num_segments=16, speed=0.2)
        
        # 打印统计
        navigator.print_summary()
        
        return True
        
    except KeyboardInterrupt:
        print("\n用户中断")
        return False
    except Exception as e:
        print(f"\n❌ 程序出错: {e}")
        import traceback
        traceback.print_exc()
        return False


if __name__ == "__main__":
    success = main()
    sys.exit(0 if success else 1)

## 3.7 调试与验证技巧

### 3.7.1 常见错误与解决方案

| 错误现象 | 原因 | 解决方案 |
|---------|------|---------|
| 路径不是正方形，边长不一 | 速度或时间计算错误 | 减小速度，增加time.sleep()的值 |
| 角度转过多或不足 | 角速度或时间计算错误 | 调整angular_speed参数 |
| 机器人走着走着停止 | 连续Move指令发送间隔过长 | 减少loop中的time.sleep() |
| 路径歪斜，不按预期方向 | 地面不平整或电机问题 | 在平坦地面测试；检查电池 |
| 运动突然中断 | 网络超时或命令失败 | 增加timeout值；检查网络 |






### 3.7.2 验证路径精度


In [ ]:
# 用粉笔或胶带标记，观察实际路径与理论路径的偏差

def verify_square(navigator, expected_side=1.0):
    """验证正方形精度"""
    print("【验证正方形精度】")
    print("请在地面用粉笔或胶带标记Go2的起点和路径")
    print("或使用测量工具（如卷尺）检查实际距离")
    
    # 记录起点
    input("按Enter开始走正方形...")
    
    navigator.trace_polygon(sides=4, edge_length=expected_side)
    
    # 记录终点
    print("\n走完正方形！")
    print(f"理论边长：{expected_side} m")
    print(f"理论总距离：{expected_side * 4} m")
    print(f"实际总距离（从里程表）：{navigator.total_distance:.2f} m")
    print(f"误差：{abs(navigator.total_distance - expected_side * 4):.2f} m")

### 3.7.3 调整参数以提高精度

In [ ]:
# 参数微调工具
def fine_tune_movement(navigator):
    """参数微调实验"""
    distances = [0.5, 1.0, 1.5, 2.0]  # 不同距离
    speeds = [0.1, 0.2, 0.3, 0.5]      # 不同速度
    
    print("【参数微调实验】")
    print("记录不同速度和距离下的实际表现")
    
    for distance in distances:
        for speed in speeds:
            print(f"\n走 {distance}m，速度 {speed} m/s")
            if input("继续? (y/n): ").lower() != 'y':
                return
            
            # 实际行走
            navigator.move_distance(distance, speed)
            
            # 用户反馈
            feedback = input("反馈（精确/偏多/偏少）: ")
            print(f"[记录] {distance}m @ {speed} m/s -> {feedback}")

### 3.7.4 调试输出


In [ ]:
# 添加详细的调试信息
def move_distance_debug(navigator, distance, speed=0.3):
    """带调试信息的移动"""
    duration = distance / speed
    
    print(f"[DEBUG] 计算参数:")
    print(f"  距离: {distance} m")
    print(f"  速度: {speed} m/s")
    print(f"  理论耗时: {duration:.2f} s")
    
    import time as time_module
    start_time = time_module.time()
    
    navigator.move_distance(distance, speed)
    
    actual_time = time_module.time() - start_time
    print(f"[DEBUG] 执行结果:")
    print(f"  实际耗时: {actual_time:.2f} s")
    print(f"  时间差: {abs(actual_time - duration):.2f} s")

## 3.8 关键概念总结

### 📊 三个运动参数

| 参数 | 含义 | 取值范围 | 用途 |
|------|------|---------|------|
| **vx** | 前后速度 | -3.5 ~ 3.5 m/s | 控制向前/后的速度 |
| **vy** | 左右速度 | -2.0 ~ 2.0 m/s | 控制向左/右的速度 |
| **vyaw** | 旋转角速度 | -2.0 ~ 2.0 rad/s | 控制原地旋转 |

### 📐 关键公式

```
运动时间 = 距离 / 速度
例：走1米，速度0.5 m/s，需要 2秒

转向时间 = 角度 / 角速度
例：转90度（π/2弧度），角速度0.5 rad/s，需要 3.14秒

正N边形的外角 = (N-2) × π / N
例：正方形（4边），外角 = π/2 = 90度
```

### 🎯 最佳实践

✅ **应该做：**
- 在Move指令后添加 `time.sleep()` 确保持续运动
- 运动前让机器人 `StandUp()`
- 检查返回值确认命令执行成功
- 在平坦、开阔的地面测试
- 使用较低的速度（0.2-0.3 m/s）以获得精度

❌ **不应该做：**
- 没有停止指令就立即发送新指令
- 使用过高的速度（容易打滑）
- 在地面不平的地方测试
- 忽视time.sleep()的间隔时间
- 假设计算是100%精确的（总有偏差）

### 🔧 精度优化

```
1. 选择合适的速度（太快容易滑，太慢不稳定）
2. 多次测试记录实际表现
3. 根据偏差调整时间系数
4. 在相同地面、相同电池电量下测试
5. 使用标记或测量工具验证结果
```

---

## 3.9 练习题

### 练习 1：直线运动精度测试

编写程序让Go2走一条2米长的直线，使用卷尺测量实际距离。

要求：
- 误差 < 0.2米
- 记录所用的速度和时间
- 打印理论值和实际值

**提示：** 使用 `move_distance()` 函数或直接调用 `Move()`。

### 练习 2：完美转向

编写程序让Go2原地转过多个角度（90°、180°、270°、360°），确保最终方向正确。

要求：
- 每个转向误差 < 5°
- 转360°后回到初始方向
- 打印每次转向前后的"方向"（虽然机器人没有方向传感器，但你可以通过标记来追踪）

### 练习 3：走五边形

编写程序让Go2走一个边长为1米的正五边形。

要求：
- 使用正确的内角（108°）和外角（72°）
- 打印每条边和每次转向的信息
- 最终回到起点附近

### 练习 4：改进的正方形

基于原有的 `move_square()` 函数，添加：
- 参数验证（检查边长是否合理）
- 错误处理（如果某步失败，打印警告）
- 总里程统计
- 起点和终点的偏差检查

### 练习 5：自定义多边形

编写一个通用函数 `trace_custom_polygon(sides, edge_length, ...)`，能够走任意正多边形。

要求：
- 支持 3-8 边形
- 自动计算内角和外角
- 添加进度提示
- 处理无效输入

### 练习 6：时间校准

由于网络延迟或电池电压差异，Move指令的实际执行时间可能与理论值不同。

编写程序来：
1. 记录多次运动的理论时间和实际时间
2. 计算修正系数（实际/理论）
3. 使用修正系数改进后续运动的精度

**提示：** 使用 `time.time()` 精确测量实际耗时。

---

## 3.10 拓展思考

### 更高级的应用

**1. 动态路径规划**
```
如果Go2能够"看见"障碍物（通过摄像头），它能否自动规划路径？
（提示：在Lesson06-07中会涉及）
```

**2. 多机器人编队**
```
多台Go2能否按照特定编队行走？
```

**3. 自适应步长**
```
根据地面条件自动调整速度和加速度？
```

**4. 基于GPS的导航**
```
（高级应用）如果有GPS，能否实现自主导航到指定坐标？
```

---

### 🎓 恭喜完成Lesson03！

你已经掌握：
✓ 三维运动坐标系统（vx, vy, vyaw）
✓ 距离与时间的关系
✓ 精确的运动控制（直线、转向、复合）
✓ 高级路径规划（正多边形、圆形、8字形等）
✓ 路径验证和精度优化
✓ 完整的工程实现和调试

**下一步：** 进入Lesson04，学习机器狗会如何跳舞！🕺